# Quantize Supertonic-3 on Colab

End-to-end, self-contained: pull upstream fp32 → quantize → verify → upload to `ahk-d/supertonic-3`.

**Runtime:** CPU is fine and is what you want — quantization is CPU-bound, GPU wastes credits.

**One prereq:** set a Colab secret called `HF_TOKEN` (sidebar key icon → Add new secret → toggle "Notebook access") with write access to `huggingface.co/ahk-d/supertonic-3`.

Then just run the cells top-to-bottom.

In [ ]:
# 1. Install Python deps.
!pip install --quiet onnx==1.21.0 onnxruntime==1.23.2 onnxconverter-common==1.16.0 numpy huggingface_hub

In [ ]:
# 2. Write tools/quantize.py inline. Mirrors react-native-speechkit/tools/quantize.py.
import pathlib, textwrap
pathlib.Path('/content/tools').mkdir(exist_ok=True)

pathlib.Path('/content/tools/quantize.py').write_text(textwrap.dedent('''\
    #!/usr/bin/env python3
    """fp16-convert Supertonic-3 ONNX weights. Attention sub-graphs are blocked
    by node name (onnxconverter_common rewrites their Cast nodes inconsistently
    and ORT refuses to load the result). keep_io_types=True so graph I/O
    remains float32 and the Swift/Kotlin loaders need no change.

    Int8 was evaluated and dropped: -1 dB SNR vs fp32 with the MatMul-only
    recipe required for iOS CPU EP compatibility.
    """
    from __future__ import annotations
    import argparse, sys
    from pathlib import Path
    import onnx
    from onnx import ModelProto
    from onnxconverter_common import float16

    ALL_MODELS = [
        "duration_predictor.onnx",
        "text_encoder.onnx",
        "vector_estimator.onnx",
        "vocoder.onnx",
    ]

    def io_signature(model: ModelProto):
        sig = []
        for tensor in list(model.graph.input) + list(model.graph.output):
            shape = tuple(
                d.dim_value if d.HasField("dim_value") else d.dim_param or "?"
                for d in tensor.type.tensor_type.shape.dim
            )
            sig.append((tensor.name, tensor.type.tensor_type.elem_type, shape))
        return sig

    def convert_fp16(src_path: Path, dst_path: Path) -> None:
        print(f"  fp16: {src_path.name}")
        model = onnx.load(str(src_path))
        pre_sig = io_signature(model)
        attn_nodes = [
            n.name for n in model.graph.node
            if "/attn/" in n.name or "/attention/" in n.name
        ]
        fp16_model = float16.convert_float_to_float16(
            model, keep_io_types=True, node_block_list=attn_nodes,
        )
        post_sig = io_signature(fp16_model)
        if pre_sig != post_sig:
            raise SystemExit(f"fp16 changed I/O signature for {src_path.name}")
        onnx.save(fp16_model, str(dst_path))
        onnx.checker.check_model(str(dst_path))
        pct = 100.0 * dst_path.stat().st_size / src_path.stat().st_size
        print(f"    -> {dst_path.stat().st_size / 1e6:6.1f} MB ({pct:.1f}% of fp32)")

    def main() -> int:
        p = argparse.ArgumentParser()
        p.add_argument("--src", type=Path, required=True)
        p.add_argument("--out", type=Path, required=True)
        a = p.parse_args()
        for n in ALL_MODELS:
            if not (a.src / n).exists():
                print(f"missing fp32 model: {a.src / n}", file=sys.stderr); return 1
        fp16_dir = a.out / "onnx-fp16"; fp16_dir.mkdir(parents=True, exist_ok=True)
        print(f"fp16 conversion ({a.src} -> {fp16_dir})")
        for n in ALL_MODELS: convert_fp16(a.src / n, fp16_dir / n)
        return 0

    if __name__ == "__main__":
        sys.exit(main())
'''))
print('wrote /content/tools/quantize.py')

In [ ]:
# 3. Write tools/verify-quantized.py inline. Mirrors react-native-speechkit/tools/verify-quantized.py.
import pathlib, textwrap

pathlib.Path('/content/tools/verify-quantized.py').write_text(textwrap.dedent('''\
    #!/usr/bin/env python3
    """Gate fp16 Supertonic-3 weights before re-hosting (I/O parity + SNR >= 20 dB)."""
    from __future__ import annotations
    import argparse, json, sys, unicodedata
    from pathlib import Path
    import numpy as np
    import onnxruntime as ort

    FP16_SNR_DB = 20.0
    REF_TEXT = "Hello, world."
    REF_LANG = "en"

    def preprocess(text, lang):
        return f"<{lang}>" + unicodedata.normalize("NFKD", text) + f"</{lang}>"

    def encode_text(processed, indexer_path):
        table = json.loads(indexer_path.read_text())
        ids = [table[ord(ch)] if ord(ch) < len(table) else -1 for ch in processed]
        return np.array([ids], dtype=np.int64)

    def load_voice(voice_path):
        blob = json.loads(voice_path.read_text())
        def comp(c):
            arr = np.array(c["data"], dtype=np.float32)
            return arr.reshape(tuple(int(d) for d in c["dims"]))
        return comp(blob["style_dp"]), comp(blob["style_ttl"])

    def session(path):
        so = ort.SessionOptions()
        so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        so.intra_op_num_threads = 2
        return ort.InferenceSession(str(path), sess_options=so, providers=["CPUExecutionProvider"])

    def io_signature(s):
        return (
            tuple((i.name, i.type, tuple(i.shape)) for i in s.get_inputs()),
            tuple((o.name, o.type, tuple(o.shape)) for o in s.get_outputs()),
        )

    def snr_db(ref, cand):
        ref = ref.astype(np.float64).ravel(); cand = cand.astype(np.float64).ravel()
        n = min(len(ref), len(cand))
        ref, cand = ref[:n], cand[:n]
        noise = ref - cand
        sp = float(np.mean(ref * ref)); np_ = float(np.mean(noise * noise))
        if np_ <= 0.0: return float("inf")
        if sp <= 0.0: return float("-inf")
        return 10.0 * np.log10(sp / np_)

    def run_pipeline(onnx_dir, text_ids, dp_style, ttl_style, cfg, seed):
        text_mask = np.ones((1, 1, text_ids.shape[1]), dtype=np.float32)
        dp_s = session(onnx_dir / "duration_predictor.onnx")
        enc_s = session(onnx_dir / "text_encoder.onnx")
        vec_s = session(onnx_dir / "vector_estimator.onnx")
        voc_s = session(onnx_dir / "vocoder.onnx")
        dp_out = dp_s.run(["duration"], {"text_ids": text_ids, "style_dp": dp_style, "text_mask": text_mask})[0]
        enc_out = enc_s.run(["text_emb"], {"text_ids": text_ids, "style_ttl": ttl_style, "text_mask": text_mask})[0]
        sr = int(cfg["ae"]["sample_rate"]); base_chunk = int(cfg["ae"]["base_chunk_size"])
        chunk_compress = int(cfg["ttl"]["chunk_compress_factor"]); latent_dim = int(cfg["ttl"]["latent_dim"]) * chunk_compress
        chunk_size = base_chunk * chunk_compress
        wav_len = int(float(dp_out.max()) * sr); latent_len = (wav_len + chunk_size - 1) // chunk_size
        latent_lengths = [(int(d * sr) + chunk_size - 1) // chunk_size for d in dp_out.ravel().tolist()]
        rng = np.random.default_rng(seed)
        noisy = rng.standard_normal((1, latent_dim, latent_len)).astype(np.float32)
        latent_mask = np.zeros((1, 1, latent_len), dtype=np.float32)
        for t in range(latent_lengths[0]): latent_mask[0, 0, t] = 1.0
        total_step = 8
        total_step_arr = np.full((1,), float(total_step), dtype=np.float32)
        for step in range(total_step):
            cur_step = np.full((1,), float(step), dtype=np.float32)
            noisy = vec_s.run(["denoised_latent"], {
                "noisy_latent": noisy, "text_emb": enc_out, "style_ttl": ttl_style,
                "latent_mask": latent_mask, "text_mask": text_mask,
                "current_step": cur_step, "total_step": total_step_arr,
            })[0]
        wav = voc_s.run(["wav_tts"], {"latent": noisy})[0]
        return {"duration": dp_out, "text_emb": enc_out, "denoised_latent": noisy, "wav_tts": wav}

    def check_io_parity(fp32_dir, other_dir, label):
        ok = True
        for name in ["duration_predictor.onnx", "text_encoder.onnx", "vector_estimator.onnx", "vocoder.onnx"]:
            ref = io_signature(session(fp32_dir / name))
            cand = io_signature(session(other_dir / name))
            if ref != cand:
                ok = False
                print(f"  [FAIL] {label} {name} I/O signature differs from fp32")
                print(f"         fp32 in:  {ref[0]}"); print(f"         {label} in:  {cand[0]}")
                print(f"         fp32 out: {ref[1]}"); print(f"         {label} out: {cand[1]}")
            else:
                print(f"  [ok]   {label} {name} I/O parity")
        return ok

    def main():
        p = argparse.ArgumentParser()
        p.add_argument("--fp32", type=Path, required=True)
        p.add_argument("--fp16", type=Path, required=True)
        p.add_argument("--indexer", type=Path, required=True)
        p.add_argument("--voice", type=Path, required=True)
        p.add_argument("--seed", type=int, default=0xC0FFEE)
        a = p.parse_args()
        print("=== I/O parity ===")
        if not check_io_parity(a.fp32, a.fp16, "fp16"):
            print("\\nI/O parity failed -- loader code would need changes. Aborting."); return 2
        cfg = json.loads((a.fp32 / "tts.json").read_text())
        text_ids = encode_text(preprocess(REF_TEXT, REF_LANG), a.indexer)
        dp_style, ttl_style = load_voice(a.voice)
        print("\\n=== Numerical regression vs fp32 ===")
        fp32 = run_pipeline(a.fp32, text_ids, dp_style, ttl_style, cfg, a.seed)
        fp16 = run_pipeline(a.fp16, text_ids, dp_style, ttl_style, cfg, a.seed)
        all_ok = True
        for stage in ["duration", "text_emb", "denoised_latent", "wav_tts"]:
            v = snr_db(fp32[stage], fp16[stage])
            status = "[ok]  " if v >= FP16_SNR_DB else "[FAIL]"
            all_ok = all_ok and v >= FP16_SNR_DB
            print(f"  {status} fp16 {stage:>16s} SNR = {v:7.2f} dB (threshold {FP16_SNR_DB})")
        if not all_ok:
            print("\\nNumerical regression failed. Do not re-host."); return 3
        print("\\nAll checks passed. Safe to mirror."); return 0

    if __name__ == "__main__":
        sys.exit(main())
'''))
print('wrote /content/tools/verify-quantized.py')

In [ ]:
# 4. Pull the upstream fp32 bundle at the commit pinned in ModelLocator.swift.
# If you bump the locator's UPSTREAM_REVISION, also bump UPSTREAM_REV below.
import urllib.request, pathlib

UPSTREAM_REV = "724fb5abbf5502583fb520898d45929e62f02c0b"
BASE = f"https://huggingface.co/Supertone/supertonic-3/resolve/{UPSTREAM_REV}"
FP32 = pathlib.Path("/content/fp32")
(FP32 / "onnx").mkdir(parents=True, exist_ok=True)
(FP32 / "voice_styles").mkdir(parents=True, exist_ok=True)

onnx_files = [
    "duration_predictor.onnx", "text_encoder.onnx",
    "vector_estimator.onnx",   "vocoder.onnx",
    "tts.json",                "unicode_indexer.json",
]
voices = ["M1", "M2", "M3", "M4", "M5", "F1", "F2", "F3", "F4", "F5"]

def fetch(rel, dst):
    if dst.exists() and dst.stat().st_size > 0:
        print(f"  skip {rel} (already on disk)"); return
    print(f"  fetch {rel} → {dst}")
    urllib.request.urlretrieve(f"{BASE}/{rel}", dst)

for f in onnx_files: fetch(f"onnx/{f}",              FP32 / "onnx" / f)
for v in voices:     fetch(f"voice_styles/{v}.json", FP32 / "voice_styles" / f"{v}.json")

!ls -lh /content/fp32/onnx /content/fp32/voice_styles

In [ ]:
# 5. fp16 conversion. Writes /content/quantized/onnx-fp16/ (4 fp16 files).
!python /content/tools/quantize.py --src /content/fp32/onnx --out /content/quantized

print("\nResulting sizes:")
!du -b /content/fp32/onnx/*.onnx           | sort -k2
!du -b /content/quantized/onnx-fp16/*.onnx | sort -k2

In [ ]:
# 6. Hard gate: I/O parity + per-stage SNR vs fp32. Exits non-zero → do not upload.
#    Threshold: fp16 ≥ 20 dB (see verify-quantized.py header for why 20 not 35).
!python /content/tools/verify-quantized.py \
    --fp32 /content/fp32/onnx \
    --fp16 /content/quantized/onnx-fp16 \
    --indexer /content/fp32/onnx/unicode_indexer.json \
    --voice   /content/fp32/voice_styles/F1.json

In [ ]:
# 7. Upload to ahk-d/supertonic-3. Requires HF_TOKEN Colab secret with write access.
# Only onnx-fp16/ is uploaded; existing onnx/ (fp32) is untouched.
from google.colab import userdata
from huggingface_hub import HfApi

api = HfApi(token=userdata.get("HF_TOKEN"))

print("Uploading onnx-fp16/ ...")
commit = api.upload_folder(
    folder_path="/content/quantized/onnx-fp16",
    path_in_repo="onnx-fp16",
    repo_id="ahk-d/supertonic-3",
    repo_type="model",
    commit_message="Add fp16 weights (attention in fp32; ~50% size cut)",
)
print(f"  commit: {commit.commit_url}")

info = api.repo_info("ahk-d/supertonic-3")
print(f"\nNew mirror revision: {info.sha}")
print("Paste this SHA into:")
print("  ios/Supertonic/ModelLocator.swift  → mirrorRevision")
print("  android/.../ModelLocator.kt        → MIRROR_REVISION")

In [ ]:
# 8. Print SHA-256 fingerprints in Swift + Kotlin paste-in form.
# Go into the placeholder block in expectedHashes / EXPECTED_HASHES.
import hashlib, pathlib

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 16), b""): h.update(chunk)
    return h.hexdigest()

MODELS = ["duration_predictor.onnx", "text_encoder.onnx", "vector_estimator.onnx", "vocoder.onnx"]

print("// --- Swift form (paste into expectedHashes in ModelLocator.swift) ---")
for f in MODELS:
    p = pathlib.Path(f"/content/quantized/onnx-fp16/{f}")
    print(f'        "onnx-fp16/{f}": "{sha256_file(p)}",')

print("\n// --- Kotlin form (paste into EXPECTED_HASHES in ModelLocator.kt) ---")
for f in MODELS:
    p = pathlib.Path(f"/content/quantized/onnx-fp16/{f}")
    print(f'    "onnx-fp16/{f}" to "{sha256_file(p)}",')